# HW5: Evaluate Before You Automate

**COMPSS 211A | Fall 2026 | Student copy**

Researchers increasingly use LLMs to label text instead of reading it all themselves. But how do you know the labels are any good? In this homework you ask Gemini to predict the verdict on 40 real r/AmItheAsshole posts, compare its answers with what the Reddit community decided, and look closely at where and why it goes wrong.

**Due:** Sunday, November 22, 2026 at 11:59 p.m.

You'll write the code that calls Gemini, reads its replies, and scores them. When a cell asks for a prediction, write your guess **before** you run anything.

Keep the variable names we ask for, because we look for them when grading.

## The data

[r/AmItheAsshole](https://www.reddit.com/r/AmItheAsshole/) ("AITA") is a Reddit forum where people describe a conflict from their own life and ask strangers whether they were in the wrong. Commenters reply with a verdict, and after about a day the post gets a label (a *flair*) based on the top-voted comment:

| Flair | Short form | Meaning |
| --- | --- | --- |
| Not the A-hole | NTA | The poster was not in the wrong. |
| Asshole | YTA | "You're the asshole": the poster was in the wrong. |
| Everyone Sucks | ESH | Everyone involved behaved badly. |
| No A-holes here | NAH | Nobody behaved badly; it's a real disagreement. |

The file `data/aita_top_subs.csv` has 5,000 real posts from 2018 to 2021. They are among the **most upvoted** posts on the forum, so they are not a random sample of AITA, let alone of Reddit or the public. The file has no usernames. Even so, these are real people's stories: don't try to find out who wrote them, and don't quote them outside this class.

You used the same posts in HW4. This time the question is different: **can an LLM predict how the community judged a post?** Keep in mind that the flair is the result of a vote among commenters, not an objective truth about who was right.

## What you need

- **Your own Gemini API key in your `.env` file**, set up in Lab 10 (instructions in `SETUP.md`, under "API keys"). In Colab, use a Colab Secret named `GEMINI_API_KEY` instead. Never paste the key into the notebook.
- Run the check in Task 3 early. If it says no key was found, fix that first.
- The 40 calls cost a few cents at most. Your results are saved to a file after the first run, so you only need to call Gemini once.

If your key doesn't work, ask on Slack or come to office hours **before** the deadline.

## What you will practice

- Setting up simple baselines before judging a model.
- Writing and testing a function that reads an LLM's reply (Week 7 tests).
- Calling an LLM in a loop and saving the results.
- Scoring predictions with accuracy and a confusion table (Week 10).
- Reading the mistakes, and working out what labeling a whole dataset would cost.

## AI and collaboration policy

Gemini is required for Task 3. Beyond that, you may use AI to explain an error message if you say so in a note in your notebook. Write your code, your notes, and your memo yourself.

In [ ]:
from pathlib import Path
import json
import os
import sys
import time

import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

SUPPORTED_PYTHON = (3, 13)
if sys.version_info[:2] != SUPPORTED_PYTHON:
    print(
        "Setup note: this course is tested with Python 3.13; "
        f"you are running {sys.version.split()[0]}."
    )

def find_course_root():
    """Find the cloned repository when this notebook is running locally."""
    for folder in (Path.cwd(), *Path.cwd().parents):
        if (folder / "pyproject.toml").exists() and (folder / "data").is_dir():
            return folder
    return None

LOCAL_COURSE_ROOT = find_course_root()
COURSE_ROOT = LOCAL_COURSE_ROOT or Path.cwd()
DATA_DIR = (
    LOCAL_COURSE_ROOT / "data"
    if LOCAL_COURSE_ROOT
    else COURSE_ROOT / "compss211_data"
)
GENERATED_DIR = COURSE_ROOT / "generated"
DATA_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

DATA_BASE_URL = (
    "https://raw.githubusercontent.com/"
    "macss-berkeley/compss-211a/main/data"
)

def course_data_path(filename):
    """Use a local course file, or download it when running in Colab."""
    path = DATA_DIR / filename
    if not path.exists():
        from urllib.request import urlretrieve

        urlretrieve(f"{DATA_BASE_URL}/{filename}", path)
    return path

print(f"Python {sys.version.split()[0]} | data={DATA_DIR}")

## 1. Start with baselines

Before you judge a model, ask: how well could you do without it?

The simplest baseline is to **always guess the most common verdict**. In this data that's NTA.

The cell below loads the posts (the same filter as HW4) and picks an **evaluation sample**: 10 posts of each verdict, 40 in total. Everyone in the class gets the same 40 posts because `random_state` is fixed. The other posts become the **training** set for the traditional classifier in Task 4.

**Your turn:**

1. Compute `majority_accuracy_all`: the accuracy of always guessing NTA on **all** posts. (Hint: `(posts["verdict"] == "NTA").mean()`.)
2. Compute `majority_accuracy_sample`: the same thing on `evaluation_sample`.
3. Explain the difference between the two numbers in a comment. Why did we build the sample with 10 of each verdict instead of picking 40 posts at random?
4. **Predict:** what accuracy do you think Gemini will get on the evaluation sample? Write it down as a comment.

In [ ]:
VERDICT_LABELS = {
    "Not the A-hole": "NTA",
    "not the a-hole": "NTA",
    "Asshole": "YTA",
    "Everyone Sucks": "ESH",
    "No A-holes here": "NAH",
}
ALLOWED_LABELS = {"NTA", "YTA", "ESH", "NAH"}

raw_posts = pd.read_csv(course_data_path("aita_top_subs.csv"))
keep = (
    raw_posts["flair_text"].isin(VERDICT_LABELS)
    & raw_posts["selftext"].notna()
    & ~raw_posts["selftext"].isin(["[removed]", "[deleted]"])
)
posts = raw_posts.loc[keep, ["idstr", "title", "selftext", "flair_text"]].copy()
posts["verdict"] = posts["flair_text"].map(VERDICT_LABELS)

evaluation_sample = posts.groupby("verdict").sample(n=10, random_state=211)
training_posts = posts.drop(evaluation_sample.index)

print(len(posts), "posts;", len(evaluation_sample), "in the evaluation sample;", len(training_posts), "for training")
posts["verdict"].value_counts()

In [ ]:
# Your turn
majority_accuracy_all = ...
majority_accuracy_sample = ...

# Why the two numbers differ, and why we balanced the sample:
#

# My prediction for Gemini's accuracy on the evaluation sample:
#

## 2. Write the reply parser, and test it

Here is the prompt you'll send. It asks Gemini to answer on one line, in the form `LABEL | reason`, for example:

```
YTA | You lashed out at your daughter for going to her father for comfort.
```

LLMs don't always follow instructions, so your code has to read the reply carefully. **Your turn:**

1. Write `parse_reply(reply)`. It should return a dictionary with two keys:
   - `llm_label`: the label in upper case, **or `None`** if the label isn't one of `ALLOWED_LABELS` or there's no `|` in the reply.
   - `llm_reason`: the text after the `|`, with spaces trimmed (or `None` if there's no `|`).

   Hints: `reply.split("|", 1)` splits at the first `|` only. `.strip()` removes spaces at both ends, and `.upper()` makes text upper case.

2. Finish the tests. The first one is done for you. Each test is one `assert` that checks one kind of reply.

In [ ]:
PROMPT_TEMPLATE = """Here is a post from the Reddit forum r/AmItheAsshole, where people describe a conflict and ask whether they were in the wrong.

Predict the verdict the Reddit community gave. Use exactly one of these labels:
NTA (not the asshole), YTA (you're the asshole), ESH (everyone sucks here), NAH (no assholes here).

Reply on one line in this format: LABEL | one-sentence reason

Title: {title}

{text}"""

In [ ]:
def parse_reply(reply):
    """Read a 'LABEL | reason' reply. Return llm_label (or None) and llm_reason."""
    # Your turn: replace the line below with your code.
    raise NotImplementedError("Write parse_reply")


# Tests
# 1 (done for you): a normal reply.
assert parse_reply("NTA | You set a fair boundary.") == {"llm_label": "NTA", "llm_reason": "You set a fair boundary."}

# 2 (your turn): extra spaces and lower case, like "  yta |  You were rude. "


# 3 (your turn): a label that isn't allowed, like "MAYBE | Hard to say."


# 4 (your turn): a reply with no "|" at all, like "I think NTA."


print("All tests passed.")

## 3. Ask Gemini

The first cell below looks for your key and prints whether it found it (never the key itself). `ask_gemini` sends one prompt and returns Gemini's reply along with how many **tokens** it used. A token is a chunk of text, roughly three-quarters of a word. There are three kinds:

- **input tokens**: your prompt, including the post;
- **output tokens**: the reply you see;
- **thinking tokens**: text the model writes to itself before answering. You never see it, but you pay for it.

**Your turn:** finish `label_posts(sample, client)`. For each post in `sample`, it should:

1. fill in `PROMPT_TEMPLATE` with the post's title and text (`PROMPT_TEMPLATE.format(title=..., text=...)`);
2. call `ask_gemini`;
3. read the reply with your `parse_reply`;
4. add one dictionary to `rows` with `idstr`, `verdict`, `raw_reply`, the token counts, and the parsed label and reason (hint: `{**a, **b}` merges two dictionaries).

The cell after it runs your function once and saves the results to `generated/hw5_gemini_labels.csv`. If that file already exists, it loads it instead of calling Gemini again.

In [ ]:
GEMINI_MODEL = "gemini-3.6-flash"  # use the replacement model if bCourses announces one


def load_api_key(name):
    """Look for an API key in .env, then your environment, then Colab Secrets.

    Returns None if there isn't one. Never prints the key.
    """
    try:
        from dotenv import find_dotenv, load_dotenv
        env_file = find_dotenv(usecwd=True)  # searches this folder, then the folders above it
        if env_file:
            load_dotenv(env_file)
    except ImportError:
        pass
    key = os.getenv(name)
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get(name)
        except Exception:
            key = None
    if key and key.startswith("paste-"):
        key = None  # the placeholder from .env.example, not a real key
    return key


print("Gemini key found:", load_api_key("GEMINI_API_KEY") is not None)


def ask_gemini(client, prompt):
    """Send one prompt. Return the reply text and its token counts."""
    for attempt in range(3):
        try:
            response = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
            usage = response.usage_metadata
            return {
                "raw_reply": response.text,
                "input_tokens": usage.prompt_token_count or 0,
                "output_tokens": usage.candidates_token_count or 0,
                "thinking_tokens": usage.thoughts_token_count or 0,
            }
        except Exception as error:
            print(f"Attempt {attempt + 1} failed: {type(error).__name__}. Retrying...")
            time.sleep(5 * (attempt + 1))
    raise RuntimeError("Gemini did not answer after three tries.")

In [ ]:
def label_posts(sample, client):
    """Ask Gemini about every post in sample. Return one row per post."""
    rows = []
    for post in sample.itertuples():
        # Your turn: build the prompt, ask Gemini, parse the reply, and append a dictionary to rows.
        ...
        print(len(rows), "of", len(sample), "done")
    return pd.DataFrame(rows)

In [ ]:
LABELS_PATH = GENERATED_DIR / "hw5_gemini_labels.csv"

if LABELS_PATH.exists():
    live_results = pd.read_csv(LABELS_PATH)
    print("Loaded saved results from", LABELS_PATH.name)
else:
    from google import genai
    key = load_api_key("GEMINI_API_KEY")
    if not key:
        raise RuntimeError("No Gemini key found. Add it to your .env file (see SETUP.md, 'API keys').")
    client = genai.Client(api_key=key)
    live_results = label_posts(evaluation_sample, client)
    live_results.to_csv(LABELS_PATH, index=False)
    print("Saved results to", LABELS_PATH.name)

print(live_results["llm_label"].isna().sum(), "replies could not be parsed")
live_results[["idstr", "verdict", "llm_label", "llm_reason"]].head()

## 4. Score the predictions

You have three methods to compare on the same 40 posts:

- **Always NTA**: the baseline from Task 1.
- **Traditional**: a TF-IDF and logistic regression classifier like the one in Week 10, trained on the other ~3,800 posts. The cell below builds it for you. (`class_weight="balanced"` tells it to take the rare verdicts seriously. Without it, the classifier just learns to say NTA every time.)
- **LLM**: Gemini's labels from Task 3.

**Your turn:**

1. Add a column `traditional_prediction` to `live_results` by matching on `idstr` (hint: `merge` with the evaluation sample's `idstr` and `traditional_prediction` columns).
2. Make `evaluation_summary`, a table with the columns `method` and `accuracy` for the three methods.
3. Make `confusion_table` for Gemini with `pd.crosstab(live_results["verdict"], live_results["llm_label"])`. Rows are the community's verdict and columns are Gemini's.
4. Make `accuracy_by_verdict`: Gemini's accuracy for each verdict (hint: a True/False column, then `groupby("verdict")`).

Your numbers may differ a little from your classmates'. Gemini doesn't always give the same answer to the same prompt, so two runs on the same 40 posts can score a few points apart. Keep that in mind when you judge how big a difference is.

In [ ]:
baseline_vectorizer = TfidfVectorizer(stop_words="english", min_df=5)
training_matrix = baseline_vectorizer.fit_transform(training_posts["title"] + " " + training_posts["selftext"])
baseline_model = LogisticRegression(max_iter=1000, class_weight="balanced")
baseline_model.fit(training_matrix, training_posts["verdict"])

evaluation_matrix = baseline_vectorizer.transform(evaluation_sample["title"] + " " + evaluation_sample["selftext"])
evaluation_sample["traditional_prediction"] = baseline_model.predict(evaluation_matrix)
evaluation_sample["traditional_prediction"].value_counts()

In [ ]:
# Your turn

## 5. Read the mistakes

Numbers only take you so far. **Your turn:**

1. Make `llm_errors`: the rows of `live_results` where Gemini's label differs from the community's verdict. Add the post `title` (merge with `evaluation_sample`) and keep `idstr`, `title`, `verdict`, `llm_label`, and `llm_reason`.
2. Read at least three mistakes, including the full post (`evaluation_sample.loc[evaluation_sample["idstr"] == "...", "selftext"].iloc[0]`). Fill in the table below. Sometimes you may think Gemini had a point. That's worth writing down: the community's verdict is a vote, not a fact.

In [ ]:
# Your turn

### Your notes on three mistakes

| idstr | Community | Gemini | Why I think Gemini disagreed | Who do I agree with? |
| --- | --- | --- | --- | --- |
|  |  |  |  |  |
|  |  |  |  |  |
|  |  |  |  |  |

## 6. What would it cost to label everything?

Suppose the research team wants Gemini labels for **all** posts in `posts`. Use your own token counts to estimate the cost.

**Your turn:**

1. Compute the average input, output, and thinking tokens per post from `live_results`.
2. Multiply by the number of posts to estimate total tokens, then turn that into dollars with the example prices below. Save the result as `cost_estimate` (a dictionary or a small table).
3. What share of the output cost comes from thinking tokens you never see?

The prices below are **examples** for the arithmetic. Check Google's [Gemini pricing page](https://ai.google.dev/gemini-api/docs/pricing) for real ones. Thinking tokens are billed at the output price.

In [ ]:
input_price_per_million = 0.50   # US dollars, example only
output_price_per_million = 3.00  # US dollars, example only; thinking tokens use this price too

# Your turn
cost_estimate = ...

## 7. Should the team use Gemini to label AITA posts?

Write five to seven sentences. Include:

1. **The numbers.** How did Gemini compare with the two baselines? How close was your prediction from Task 1? With 40 posts, how much would one or two answers change the accuracy?
2. **The pattern.** Which verdicts does Gemini get right, and which does it miss? Use `confusion_table` or `accuracy_by_verdict`.
3. **Two real mistakes.** Refer to two posts from your notes by `idstr`.
4. **What the label means.** The "right answer" here is a Reddit vote. What does that mean for how you read Gemini's accuracy?
5. **Cost and trust.** Using your `cost_estimate`, is labeling all posts affordable? What check would you want before trusting Gemini's labels in a research paper?

### Your response

**Question:** Based on your evidence, should the team use Gemini to label AITA posts, and what would make you pause?

> Write your response here, then delete this line.